<a href="https://colab.research.google.com/github/CSSB-SNU/Thal-Kak_for_release/blob/main/Thalkak.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thal-Kak

End-to-end protein structure prediction: **MSA** (ColabFold) → **structure** (Boltz-2 / Chai-1 / Protenix / ESMFold2) → **Amber relaxation**, with per-model confidence-based top-5 selection.

**How to run:** set your inputs in step 1, then `Runtime` → `Run all`.

> **Requirements**
> - A **GPU** runtime: `Runtime` → `Change runtime type` → **GPU** (T4 is fine).
> - Protein targets only (RNA needs a large local database built offline).
> - First run installs a full conda environment and downloads model weights, so expect **~20–30 min** before results appear.

In [ ]:
#@title 1. Clone the Thal-Kak repository
import os, subprocess

REPO = "CSSB-SNU/Thal-Kak_for_release"
REPO_DIR = "/content/Thal-Kak_for_release"

if not os.path.isdir(REPO_DIR):
    rc = subprocess.run(
        ["git", "clone", "--depth", "1",
         f"https://github.com/{REPO}.git", REPO_DIR]
    ).returncode
    if rc != 0:
        raise RuntimeError("git clone failed.")
    print("Cloned", REPO)
else:
    print("Repo already present, skipping clone.")

In [ ]:
#@title 2. Install dependencies (Miniforge + `thalkak` env) — slow, ~20–30 min
#@markdown Installs Miniforge, then runs the repo's `install.sh` to build the
#@markdown unified `thalkak` conda environment and patch ColabFold templates.
#@markdown Flag files make re-runs cheap.
import os, shlex

script = r"""
set -e
if [ ! -f /content/CONDA_READY ]; then
  echo "== Installing Miniforge (conda/mamba) =="
  wget -qO /content/miniforge.sh https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
  bash /content/miniforge.sh -b -f -p /opt/conda
  rm -f /content/miniforge.sh
  touch /content/CONDA_READY
fi
source /opt/conda/etc/profile.d/conda.sh
if [ ! -f /content/THALKAK_READY ]; then
  echo "== Building thalkak env via install.sh (slow) =="
  cd /content/Thal-Kak_for_release
  bash install.sh
  touch /content/THALKAK_READY
fi
echo "== Install step complete =="
"""

rc = os.system("bash -c " + shlex.quote(script))
if rc != 0:
    raise RuntimeError("Installation failed — see the log above.")

In [ ]:
#@title 3. Input sequence & options
import os

jobname = "T1201" #@param {type:"string"}
#@markdown - Protein sequence. For a **complex**, separate chains with `:`.
query_sequence = "ETGCNKALCASDVSKCLIQELCQCRPGEGNCSCCKECMLCLGALWDECCDCVGMCNPRNYSDTPPTSKSTVEELHEPIPSLFRALTEGDTQLNWNIVSFPVAEELSHHENLVSFLETVNQPHHQNVSVPSNNVHAPYSSDKEHMCTVVYFDDCMSIHQCKISCESMGASKYRWFHNACCECIGPECIDYGSKTVKCMNCMFGTKHHHHHH" #@param {type:"string"}
#@markdown - Stoichiometry, e.g. `A1` (monomer), `A2` (homodimer), `A1B1`
#@markdown   (heterodimer). Leave as `UNK` to use one copy of each chain.
stoichiometry = "UNK" #@param {type:"string"}
#@markdown ---
#@markdown ### Model & run options
structure_model = "boltz2" #@param ["boltz2", "chai1", "protenix", "esmfold2"]
relax_method = "amber" #@param ["amber", "none"]
num_seeds = 5 #@param [1, 2, 5] {type:"raw"}

# Build a FASTA (one record per ":"-separated chain).
seqs = [s.strip().upper().replace(" ", "")
        for s in query_sequence.split(":") if s.strip()]
assert seqs, "query_sequence is empty"
workdir = f"/content/{jobname}"
os.makedirs(workdir, exist_ok=True)
fasta_path = f"{workdir}/{jobname}.fa"
with open(fasta_path, "w") as f:
    for i, s in enumerate(seqs):
        f.write(f">{jobname}_{chr(65 + i)}\n{s}\n")

print(f"Wrote {len(seqs)} chain(s) to {fasta_path}")
print(f"model={structure_model}  relax={relax_method}  "
      f"seeds={num_seeds}  stoi={stoichiometry}")

In [ ]:
#@title 4. Run Thal-Kak (MSA → structure → relax)
import os, shlex

cmd = (
    "source /opt/conda/etc/profile.d/conda.sh && conda activate thalkak && "
    "cd /content/Thal-Kak_for_release && "
    "python thalkak.py full --msa colab "
    f"--structure {shlex.quote(structure_model)} "
    f"--relax {shlex.quote(relax_method)} "
    f"--seq {shlex.quote(fasta_path)} "
    f"--stoi {shlex.quote(stoichiometry)} "
    f"--n_seed {int(num_seeds)} "
    f"--base_dir {shlex.quote(workdir)}"
)
rc = os.system("bash -c " + shlex.quote(cmd))
if rc != 0:
    raise RuntimeError("Thal-Kak run failed — see the log above.")
print("Done. Outputs under:", workdir)

In [ ]:
#@title 5. View the top relaxed model
import glob

pdbs = sorted(glob.glob(f"{workdir}/top5/**/relaxed/**/*.pdb", recursive=True))
if not pdbs:
    pdbs = sorted(glob.glob(f"{workdir}/top5/**/model_*.pdb", recursive=True))
print(f"Found {len(pdbs)} top-5 model(s):")
for p in pdbs:
    print("  ", p)

if pdbs:
    try:
        import py3Dmol
    except ImportError:
        os.system("pip -q install py3Dmol")
        import py3Dmol
    view = py3Dmol.view(width=700, height=500)
    view.addModel(open(pdbs[0]).read(), "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    view.show()

In [ ]:
#@title 6. Download results
#@markdown Zips the whole job folder (MSA, structures, top-5, relaxed models,
#@markdown confidence logs) and downloads it. Optionally copy to Google Drive.
save_to_google_drive = False #@param {type:"boolean"}
import shutil

zip_base = f"/content/{jobname}.result"
shutil.make_archive(zip_base, "zip", workdir)
zip_path = zip_base + ".zip"
print("Result archive:", zip_path)

if save_to_google_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = f"/content/drive/MyDrive/{jobname}.result.zip"
    shutil.copyfile(zip_path, dst)
    print("Saved to", dst)

from google.colab import files
files.download(zip_path)

## Notes

- **Models & weights.** Each backend downloads its own weights on first run (e.g. `~/.boltz` for Boltz-2, HuggingFace cache for ESMFold2). Weights are distributed under their providers' own terms.
- **Outputs.** For a job named `T1201` the results live under `/content/T1201/`: `msa/`, `structure/`, and `top5/<job>/` with `model_1..5.pdb`, `relaxed/<method>/`, and `method_log.yaml` provenance.
- **RNA/DNA.** This notebook targets proteins. RNA MSA needs a large local database (`prepare_db.sh`) that is impractical to build on Colab — run RNA targets locally instead.
- **License.** Thal-Kak is Apache-2.0; see `LICENSE` and `NOTICE` in the repo.